In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
from forestplot import forestplot
from matplotlib.ticker import FuncFormatter
import matplotlib.pyplot as plt
plt.rcParams['font.monospace'] = ['Noto Mono', 'Tlwg Mono', 'Courier New']
import seaborn as sns
sns.set_theme(context="notebook", font_scale=1.25, style="white", palette="dark")

from helpers import init_mpl_fig, save_mpl_fig

In [ ]:
df = pd.read_parquet("gc_human_tweets.parquet").assign(
    hashtags=lambda df_: df_["hashtags"].apply(
        lambda x: x.tolist() if isinstance(x, np.ndarray) else x
    )
)
df

## Top hashtags

In [ ]:
# Get all hashtags (excluding #metoo)
all_hashtags = []
for hashtags_list in df["hashtags"]:
    # Exclude #metoo variants
    other_tags = [tag for tag in hashtags_list if "metoo" not in tag]
    all_hashtags.extend(other_tags)

df_hashtags = (
    pd.DataFrame.from_dict(Counter(all_hashtags), orient="index", columns=["count"])
    .sort_values("count", ascending=False)
    .reset_index(names="hashtag")
)
df_hashtags.to_csv("output/hashtags.csv", index=False)
df_hashtags.head(20)

### tuples

In [ ]:
# Get all hashtag combinations (as tuples)
hashtag_tuples = []
for hashtags_list in df["hashtags"]:
    if len(hashtags_list) > 1:  # Only tweets with multiple hashtags
        # Sort to normalize order, convert to tuple
        hashtag_tuple = tuple(sorted(hashtags_list))
        hashtag_tuples.append(hashtag_tuple)

tuple_counts = Counter(hashtag_tuples)

print(f"Total tweets with 2+ hashtags: {len(hashtag_tuples):,}")
print(f"Unique hashtag combinations: {len(tuple_counts):,}\n")

print("TOP 10 MOST COMMON HASHTAG COMfBINATIONS")
print("=" * 80)
for i, (tags, count) in enumerate(tuple_counts.most_common(10), 1):
    print(f"\n{i}. Count: {count:,} tweets ({count/len(df)*100:.2f}%)")
    print(f"   Hashtags ({len(tags)}): {', '.join(tags)}")

## Plot

### Prep

In [ ]:
TOP_N = 30
top_hashtags = Counter(all_hashtags).most_common(TOP_N)
hashtag_metadata = []

for hashtag, count in top_hashtags:
    # Get all tweets containing this hashtag
    tweets_with_hashtag = df[
        df["hashtags"].apply(lambda x: hashtag in x if isinstance(x, list) else False)
    ]

    # Calculate metrics
    pct = (count / len(df)) * 100
    mean_retweets = tweets_with_hashtag["retweets"].mean()
    mean_favs = tweets_with_hashtag["favorites"].mean()

    # Calculate mean number of hashtags per tweet
    hashtag_counts = tweets_with_hashtag["hashtags"].apply(
        lambda x: len(x) if isinstance(x, list) else 0
    )
    mean_hashtags = hashtag_counts.mean()

    hashtag_metadata.append(
        {
            "hashtag": hashtag,
            "count": count,
            "pct": pct,
            "mean_hashtags": mean_hashtags,
            "mean_retweets": mean_retweets,
            "mean_favs": mean_favs,
        }
    )

df_plot = pd.DataFrame(hashtag_metadata)

# Create formatted strings with proper alignment
df_plot["count_str"] = df_plot["count"].apply(lambda x: f"{x:,}".rjust(6))
df_plot["pct_str"] = df_plot["pct"].apply(lambda x: f"{x:.1f}%".rjust(8))
df_plot["rt_str"] = df_plot["mean_retweets"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["fav_str"] = df_plot["mean_favs"].apply(lambda x: f"{x:.1f}".rjust(4))
df_plot["ht_str"] = df_plot["mean_hashtags"].apply(lambda x: f"{x:.1f}".rjust(5))

# For forestplot - use pct as both estimate and limits (creates dot plot)
df_plot["ll"] = df_plot["pct"]
df_plot["hl"] = df_plot["pct"]

df_plot[
    ["hashtag", "count", "pct", "mean_retweets", "mean_favs", "mean_hashtags"]
].head(10)

df_plot

### plot

In [ ]:
ax = forestplot(
    df_plot,
    estimate="pct",  # What to plot (percentage)
    ll="ll",  # Lower limit (same as estimate for dot plot)
    hl="hl",  # Upper limit (same as estimate for dot plot)
    varlabel="hashtag",  # Y-axis labels (hashtag names)
    annote=["rt_str", "fav_str"],  # Left-side annotations
    annoteheaders=[" RT", "Fave"],  # Left annotation headers
    rightannote=["count_str", "pct_str"],  # Right-side annotations
    right_annoteheaders=["Tweets", "Tweets(%)"],  # Right annotation headers
    ci_report=False,  # No confidence intervals
    figsize=(5, TOP_N * 0.4),  # Scale height with number of hashtags
    color_alt_rows=True,  # Alternate row colors for readability
    table=True,  # Show as table
    xticks=range(0, 10),  # X-axis ticks (adjust based on your max %)
    **{
        "marker": "o",
        "variable_header": "Hashtag",
        "markersize": 40,
        "xlinecolor": ".8",
        # Font size controls
        "xtick_size": 11,  # X-axis tick numbers
    },
)

# Format x-axis as percentages
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{int(x)}%"))
ax.set_xlabel("Percentage of #MeToo tweets", fontsize=12)

# Add subtle gridlines at major percentages
for vline in [1, 2, 3, 4, 5, 6, 7, 8, 9]:
    if vline % 2 == 0:
        ax.vlines(vline, ymin=-0.5, ymax=TOP_N - 0.5, colors=".85", linewidth=0.5)

save_mpl_fig("output/figures/top_hashtags_tabulation")

## hashtag density

In [ ]:
# Plot n_hashtags distribution
x = df["n_hashtags"].dropna().to_numpy()

ks = np.arange(1, 6)
pk = [(x <= k).mean() for k in ks]

_, ax = init_mpl_fig(aspect_ratio=(6, 4), scale=1.4)

sns.ecdfplot(
    data=df,
    x="n_hashtags",
    color="0.3",
    linewidth=1.6,
    ax=ax,
)

lines = [f"P(n ≤ {k}) = {p:>4.1%}" for k, p in zip(ks, pk)]
txt = "\n".join(lines)

ax.text(
    0.97,
    0.08,
    txt,
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=13,
    linespacing=1.6,
    fontfamily="DejaVu Sans",
    color="0.2",
    bbox=dict(facecolor="white", edgecolor="none", alpha=0.85),
)

ax.set_xlabel("Number of hashtags")
ax.set_ylabel("Empirical CDF")
ax.set_ylim(0, 1)
plt.tight_layout()
save_mpl_fig("output/figures/hashtag_density_ecdf")